In [1]:
import torch
import pandas as pd
from pathlib import Path
from pykeen.triples import TriplesFactory
from pykeen import predict
from neo4j import GraphDatabase

PROJECT_ROOT = Path.home() / "xai-knowledge-graph"
MODEL_DIR    = PROJECT_ROOT / "models" / "rotate"

# Load trained model
model = torch.load(MODEL_DIR / "trained_model.pkl", map_location="cpu", weights_only=False)
model.eval()

# Load the triples factory (entity-to-ID mapping)
tf = TriplesFactory.from_path_binary(str(MODEL_DIR / "training_triples"))
print(f"Model loaded — {tf.num_entities:,} entities, {tf.num_relations} relations")

# Connect to Neo4j to fetch human-readable titles/names
def load_creds(path=str(Path.home() / "aura_cred.txt")):
    creds = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if "=" in line and not line.startswith("#"):
                k, v = line.split("=", 1)
                creds[k.strip()] = v.strip()
    return creds

creds  = load_creds()
driver = GraphDatabase.driver(creds["NEO4J_URI"], auth=(creds["NEO4J_USERNAME"], creds["NEO4J_PASSWORD"]))
DB     = creds["NEO4J_DATABASE"]

Model loaded — 18,603 entities, 4 relations


In [2]:
print("Building entity → display-name lookup from Neo4j...")
entity_labels = {}
with driver.session(database=DB) as s:
    for r in s.run("MATCH (p:Paper) RETURN p.arxiv_id AS aid, p.title AS title"):
        entity_labels[f"paper:{r['aid']}"] = r["title"]
    for r in s.run("MATCH (a:Author) RETURN a.name AS name"):
        entity_labels[f"author:{r['name']}"] = r["name"]
    for r in s.run("MATCH (t:Topic) RETURN t.name AS name"):
        entity_labels[f"topic:{r['name']}"] = r["name"]
    for r in s.run("MATCH (v:Venue) RETURN v.name AS name"):
        entity_labels[f"venue:{r['name']}"] = r["name"]

def label(entity_id):
    return entity_labels.get(entity_id, entity_id)

print(f" Loaded {len(entity_labels):,} display labels")

Building entity → display-name lookup from Neo4j...
 Loaded 18,603 display labels


In [3]:
def find_similar_entities(entity_label, k=10, prefix=None):
    if entity_label not in tf.entity_to_id:
        print(f" Entity not found: {entity_label}")
        return None

    embs = model.entity_representations[0]().detach().cpu()
    
    # ─── handle complex-valued embeddings (RotatE) ───
    if embs.is_complex():
        embs = torch.cat([embs.real, embs.imag], dim=-1)
    # ──────────────────────────────────────────────────────
    
    query_id = tf.entity_to_id[entity_label]
    query_emb = embs[query_id]

    sims = torch.nn.functional.cosine_similarity(query_emb.unsqueeze(0), embs, dim=1)

    id_to_entity = {v: k_ for k_, v in tf.entity_to_id.items()}
    ranked = sorted(enumerate(sims.tolist()), key=lambda x: -x[1])

    results = []
    for idx, sim in ranked:
        if idx == query_id:
            continue
        ent = id_to_entity[idx]
        if prefix and not ent.startswith(prefix):
            continue
        results.append((ent, sim))
        if len(results) >= k:
            break
    return results

In [4]:
TARGET_ID = "paper:2105.07190"
print(f"=== Top 10 papers most similar to: {label(TARGET_ID)} ===\n")

similar = find_similar_entities(TARGET_ID, k=10, prefix="paper:")
for ent, sim in similar:
    print(f"  [{sim:.3f}]  {label(ent)[:80]}")

=== Top 10 papers most similar to: A Comprehensive Taxonomy for Explainable Artificial Intelligence: A Systematic Survey of Surveys on Methods and Concepts ===

  [0.497]  Explainable Activity Recognition for Smart Home Systems
  [0.469]  Resisting Out-of-Distribution Data Problem in Perturbation of XAI
  [0.462]  Explainable AI (XAI): A Systematic Meta-Survey of Current Challenges and Future 
  [0.454]  Human-Centered Explainable AI (XAI): From Algorithms to User Experiences
  [0.438]  From Anecdotal Evidence to Quantitative Evaluation Methods: A Systematic Review 
  [0.412]  Beyond Additivity: Sparse Isotonic Shapley Regression toward Nonlinear Explainab
  [0.407]  A Theoretical Framework for AI Models Explainability with Application in Biomedi
  [0.407]  Causality-Inspired Taxonomy for Explainable Artificial Intelligence
  [0.404]  From Black Boxes to Conversations: Incorporating XAI in a Conversational Agent
  [0.396]  RobustX: Robust Counterfactual Explanations Made Easy


In [6]:
print(f"=== Predicted citation targets for: {label(TARGET_ID)} ===\n")

pred = predict.predict_target(
    model=model,
    head=TARGET_ID,
    relation="cites",
    triples_factory=tf,
)
pred_df = pred.df

# Filter to only paper entities (not authors, topics, venues)
pred_df = pred_df[pred_df["tail_label"].str.startswith("paper:")]
top_predictions = pred_df.head(10)

print(f"{'Score':>7}  Title")
print("-" * 90)
for _, row in top_predictions.iterrows():
    print(f"{row['score']:>7.3f}  {label(row['tail_label'])[:80]}")

=== Predicted citation targets for: A Comprehensive Taxonomy for Explainable Artificial Intelligence: A Systematic Survey of Surveys on Methods and Concepts ===

  Score  Title
------------------------------------------------------------------------------------------
 -7.004  Explainability in Deep Reinforcement Learning
 -7.028  Explainable Artificial Intelligence Approaches: A Survey
 -7.179  The Need for Standardized Explainability
 -7.243  Explainable Reinforcement Learning: A Survey
 -7.293  Quantitative Evaluations on Saliency Methods: An Experimental Study
 -7.398  Opportunities and Challenges in Explainable Artificial Intelligence (XAI): A Sur
 -7.481  If Only We Had Better Counterfactual Explanations: Five Key Deficits to Rectify 
 -7.505  Explanation in Artificial Intelligence: Insights from the Social Sciences
 -8.012  Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization
 -8.420  Explainable AI (XAI): A Systematic Meta-Survey of Current Challenges

In [7]:
TARGET_AUTHOR = "author:S. Lapuschkin"
print(f"=== Authors most similar to: {label(TARGET_AUTHOR)} ===\n")

similar_authors = find_similar_entities(TARGET_AUTHOR, k=10, prefix="author:")
for ent, sim in similar_authors:
    print(f"  [{sim:.3f}]  {label(ent)}")

=== Authors most similar to: S. Lapuschkin ===

  [0.688]  Leander Weber
  [0.676]  Galip Umit Yolcu
  [0.641]  Alexander Binder
  [0.641]  Frederik Pahde
  [0.640]  Maximilian Dreyer
  [0.631]  W. Samek
  [0.627]  Thomas Wiegand
  [0.563]  Wojciech Samek
  [0.543]  Reduan Achtibat
  [0.516]  Christopher J. Anders


In [10]:
# Get Grad-CAM's actual outgoing citations from Neo4j
with driver.session(database=DB) as s:
    actual_cites = set()
    for r in s.run(
        "MATCH (p:Paper {arxiv_id: '2105.07190'})-[:CITES]->(c:Paper) RETURN c.arxiv_id AS aid"
    ):
        actual_cites.add(f"paper:{r['aid']}")

# Of the top 10 predictions, how many are actually cited?
top_10_predicted = top_predictions["tail_label"].tolist()
hits = [p for p in top_10_predicted if p in actual_cites]
print(f"Actual citations in A Comprehensive Taxonomy for Explainable Artificial Intelligence: {len(actual_cites)}")
print(f"Top 10 predictions overlap with actual: {len(hits)}/{len(top_10_predicted)}")
print(f"Precision@10 for this paper: {len(hits)/len(top_10_predicted):.2f}")

print(f"\nHit titles:")
for h in hits:
    print(f"   {label(h)[:80]}")

Actual citations in A Comprehensive Taxonomy for Explainable Artificial Intelligence: 14
Top 10 predictions overlap with actual: 9/10
Precision@10 for this paper: 0.90

Hit titles:
   Explainability in Deep Reinforcement Learning
   Explainable Artificial Intelligence Approaches: A Survey
   The Need for Standardized Explainability
   Explainable Reinforcement Learning: A Survey
   Quantitative Evaluations on Saliency Methods: An Experimental Study
   Opportunities and Challenges in Explainable Artificial Intelligence (XAI): A Sur
   If Only We Had Better Counterfactual Explanations: Five Key Deficits to Rectify 
   Explanation in Artificial Intelligence: Insights from the Social Sciences
   Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization


In [9]:
# Find papers with most outgoing in-corpus citations
with driver.session(database=DB) as s:
    result = s.run("""
        MATCH (p:Paper)-[:CITES]->(c:Paper)
        WITH p, count(c) AS out_degree
        WHERE out_degree >= 10
        RETURN p.arxiv_id AS aid, p.title AS title, p.year AS year, out_degree
        ORDER BY out_degree DESC
        LIMIT 10
    """)
    print(f"{'Out':>5}  {'Year':>4}  {'arXiv ID':<14}  Title")
    print("-" * 110)
    for r in result:
        print(f"{r['out_degree']:>5}  {r['year']:>4}  {r['aid']:<14}  {r['title'][:80]}")

  Out  Year  arXiv ID        Title
--------------------------------------------------------------------------------------------------------------
   22  2025  2510.21599      SHAP Meets Tensor Networks: Provably Tractable Explanations with Parallelism
   17  2024  2409.14590      Explainable AI needs formalization
   15  2024  2409.00265      Explainable Artificial Intelligence: A Survey of Needs, Techniques, Applications
   14  2026  2603.25251      Does Explanation Correctness Matter? Linking Computational XAI Evaluation to Hum
   14  2021  2105.07190      A Comprehensive Taxonomy for Explainable Artificial Intelligence: A Systematic S
   13  2025  2509.16685      Towards a Transparent and Interpretable AI Model for Medical Image Classificatio
   12  2026  2603.06485      PONTE: Personalized Orchestration for Natural Language Trustworthy Explanations
   12  2022  2210.03735      "Help Me Help the AI": Understanding How Explainability Can Support Human-AI Int
   11  2026  2603.16067  